# GPU-CorruptNet — train + calibrate on a free Colab T4

1. **Runtime → Change runtime type → GPU (T4)**.
2. Run all cells. Trains the classifier on STL-10 + on-the-fly Glitchify-2 corruptions, prints macro-F1 on seen- vs unseen-content test sets, then runs M5 calibration (ECE + conformal).

The backbone is **fine-tuned** (`--no-freeze`), which is what reaches strong F1. An NVIDIA GPU also makes the later latency benchmark (CUDA-event timing, FP16) authentic.

In [ ]:
!git clone https://github.com/tanaymihani/gpu-corruptnet.git
%cd gpu-corruptnet
!pip install -q -e ".[cv,train]"

In [ ]:
import torch
ok = torch.cuda.is_available()
print('cuda:', ok, '|', torch.cuda.get_device_name(0) if ok else 'NO GPU - set Runtime to T4')

In [ ]:
# ResNet-50, full-resolution, FINE-TUNED (unfrozen backbone) - this is what reaches strong F1.
!python scripts/train_classifier.py --arch resnet50 --epochs 12 --img-size 224 \
    --batch-size 64 --lr 1e-4 --no-freeze --num-workers 2

In [ ]:
# EfficientNet-B4 for the head-to-head comparison in your resume bullet.
!python scripts/train_classifier.py --arch efficientnet_b4 --epochs 12 --img-size 224 \
    --batch-size 32 --lr 1e-4 --no-freeze --num-workers 2

In [ ]:
# M5: calibration (ECE before/after temperature scaling) + conformal coverage on the latest run.
import glob
latest = sorted(glob.glob('runs/preds_*.npz'))[-1]
!python scripts/calibrate.py {latest} --alpha 0.1

In [ ]:
import glob, json
for f in sorted(glob.glob('runs/metrics_*.json')):
    m = json.load(open(f))
    print('\n' + f)
    for split in ('seen_test', 'unseen_test'):
        s = m[split]
        print('  %-12s macroF1=%.3f binaryF1=%.3f binaryRecall=%.3f'
              % (split, s['macro_f1'], s['binary_f1'], s['binary_recall']))